# 🎬 Free AI Video Generator (Wan 2.1 on Colab)

Make short animated clips **for free** using the open-source **Wan 2.1 (1.3B)** model.

**Before you start:** `Runtime → Change runtime type → T4 GPU` → Save.

Then run each cell from top to bottom (▶ button). The first run downloads the model (~5–6 minutes).


## 1. Install

In [ ]:
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg sentencepiece
import torch
assert torch.cuda.is_available(), "No GPU! Go to Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Load the model (one time per session)

The text encoder is too large for free Colab's memory, so it's loaded **only while reading your prompt** and then removed again.


In [ ]:
import torch, gc
from diffusers import AutoencoderKLWan, WanPipeline
from transformers import UMT5EncoderModel

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL_ID, vae=vae, text_encoder=None, torch_dtype=torch.bfloat16)
pipe.to("cuda")
try:
    pipe.vae.enable_tiling()
except Exception:
    pass

def encode(prompt, negative):
    """Load the big text encoder on the GPU, read the prompt, then free it."""
    pipe.transformer.to("cpu"); torch.cuda.empty_cache()
    te = UMT5EncoderModel.from_pretrained(MODEL_ID, subfolder="text_encoder",
                                          torch_dtype=torch.bfloat16, device_map="cuda")
    pipe.text_encoder = te
    with torch.no_grad():
        pe, ne = pipe.encode_prompt(prompt=prompt, negative_prompt=negative,
                                    do_classifier_free_guidance=True, device="cuda")
    pipe.text_encoder = None
    del te; gc.collect(); torch.cuda.empty_cache()
    pipe.transformer.to("cuda")
    return pe, ne

print("Model ready ✅")

## 3. Write your scene & generate

- **STYLE** is added to every prompt so all your clips look like the same series.
- **SECONDS**: 3 is fast (~8–12 min on T4). 5 works but takes longer (~20+ min).
- Change **SEED** to get a different version of the same scene.


In [ ]:
#@title Scene settings
PROMPT = "A simple cartoon character with a big round pale head lies in bed in a dark bedroom. A red alarm clock flips to 5:00. His eyes snap open, he frowns with determination and sits up, fists clenched."  #@param {type:"string"}
STYLE = "2D hand-drawn cartoon animation, bold black outlines, flat colors, dark moody lighting, expressive faces, motivational short film"  #@param {type:"string"}
SECONDS = 3  #@param {type:"slider", min:1, max:5, step:1}
VERTICAL = False  #@param {type:"boolean"}
SEED = 42  #@param {type:"integer"}
STEPS = 30  #@param {type:"slider", min:15, max:50, step:5}

NEGATIVE = ("realistic, photo, 3d render, blurry, low quality, distorted face, extra limbs, "
            "deformed hands, text, watermark, static, still image, jpeg artifacts")

FPS = 16
num_frames = SECONDS * FPS + 1
num_frames = ((num_frames - 1) // 4) * 4 + 1   # Wan needs 4k+1 frames
width, height = (480, 832) if VERTICAL else (832, 480)

import time
t0 = time.time()
print("Reading prompt...")
prompt_embeds, negative_embeds = encode(f"{STYLE}. {PROMPT}", NEGATIVE)
print("Generating video...")
frames = pipe(
    prompt_embeds=prompt_embeds,
    negative_prompt_embeds=negative_embeds,
    height=height, width=width,
    num_frames=num_frames,
    guidance_scale=5.0,
    num_inference_steps=STEPS,
    generator=torch.Generator("cpu").manual_seed(SEED),
).frames[0]
print(f"Done in {(time.time()-t0)/60:.1f} min")

## 4. Save, preview & download

In [ ]:
from diffusers.utils import export_to_video
from IPython.display import Video, display
import os

os.makedirs("clips", exist_ok=True)
out = f"clips/clip_seed{SEED}_{int(time.time())}.mp4"
export_to_video(frames, out, fps=FPS)
display(Video(out, embed=True, width=640))

from google.colab import files
files.download(out)

## 5. (Optional) Join all your clips into one video

Generate several scenes with step 3 + 4, then run this to stitch them in order.


In [ ]:
import glob, subprocess
clips = sorted(glob.glob("clips/*.mp4"), key=os.path.getmtime)
with open("list.txt", "w") as f:
    for c in clips:
        f.write(f"file '{c}'\n")
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0",
                "-i", "list.txt", "-c", "copy", "full_video.mp4"], check=True)
print("Joined", len(clips), "clips")
display(Video("full_video.mp4", embed=True, width=640))
files.download("full_video.mp4")

## Tips
- **Keep prompts to one action per clip.** Short, simple scenes look best.
- **Same STYLE every time** keeps your character consistent across a series.
- If you get an **out-of-memory** error: lower SECONDS, or `Runtime → Restart session` and run again.
- Free Colab disconnects after a while of inactivity or ~a few hours of use, so download clips as you go.
- Voiceover, captions, titles like **"DAY 1."** and music can be added afterwards (ask Claude for the editing script).
